# M0 environment probe (PLAN §7 M0, DONE DC-16)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN** (required),
**GITHUB_TOKEN** (only if the repo is private).

Run all cells top to bottom (about 30 min). Then paste back the output of the last cell plus
`env.json`, `vllm_probe.json`, `pip_check.txt` and the two pytest summaries from
`/kaggle/working/artifacts/env/<session>/` (also uploaded to the private store under `env/<session>/`).

No logic lives here: every step is a pinned-package CLI call (PLAN §3 thin-notebook contract).

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = dict(os.environ)
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
# M0-T4: record Kaggle's preinstalled stack, install the pinned extras, record the result.
!mkdir -p /kaggle/working/artifacts/env/$RBBD_SESSION_ID
!pip freeze > /kaggle/working/artifacts/env/$RBBD_SESSION_ID/pip_freeze_before.txt
!pip install -q -e ".[train,bench,dev]" 2>&1 | tee /kaggle/working/artifacts/env/$RBBD_SESSION_ID/pip_install.txt | tail -25
!pip freeze > /kaggle/working/artifacts/env/$RBBD_SESSION_ID/pip_freeze.txt
!pip check > /kaggle/working/artifacts/env/$RBBD_SESSION_ID/pip_check.txt 2>&1; tail -20 /kaggle/working/artifacts/env/$RBBD_SESSION_ID/pip_check.txt

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# M0-T1 + M0-T5: hardware, disk, RAM, torch, bf16, gated access -> env.json (fails loudly without 2x sm75).
!python -m rbbd.cli probe

In [ ]:
# DC-01/DC-02 on Kaggle's resolved stack.
!ruff check src tests 2>&1 | tee /kaggle/working/artifacts/env/$RBBD_SESSION_ID/ruff.txt | tail -3
!pytest -q -m "not gpu" 2>&1 | tee /kaggle/working/artifacts/env/$RBBD_SESSION_ID/pytest_cpu.txt | tail -5

In [ ]:
# M0-T6: vLLM hello-world (Qwen2.5-0.5B TP=1/TP=2 fp16, plain and merged weights; LoRA canary; Gemma-3-1B fp16/fp32).
!pytest -q -m gpu tests/gpu/test_generate_gpu.py::test_vllm_hello_tp1_tp2_merged 2>&1 | tee /kaggle/working/artifacts/env/$RBBD_SESSION_ID/pytest_gpu_vllm.txt | tail -15

In [ ]:
# M0-T7: private-store round trip (refuses unless the repo is private), then upload this session's env folder.
!python -m rbbd.cli probe --check-store
!python -m rbbd.cli sync --path env/$RBBD_SESSION_ID

In [ ]:
# Summary to paste back.
import json
info = json.load(open(f"{ENV_DIR}/env.json"))
vllm = json.load(open(f"{ENV_DIR}/vllm_probe.json"))["cases"] if os.path.exists(f"{ENV_DIR}/vllm_probe.json") else []
print(json.dumps({
    "session": info["session_id"], "gpus": info["gpus"], "bf16_supported": info["bf16_supported"],
    "torch": info["torch"], "ram": info["ram"], "disk": info["disk"], "ephemeral_dir": info["ephemeral_dir"],
    "requirements": info["requirements"], "access_all_ok": info["access_all_ok"],
    "access_failures": {k: v for k, v in info["access"].items() if v != "ok"},
    "store": info.get("store"),
    "vllm": [{"name": c["case"]["name"], "ok": c["ok"], "failed_stage": c.get("failed_stage"),
              "engine": c.get("engine_module"), "seconds": c.get("seconds"),
              "merge_seconds": c.get("merge_seconds"), "merged_dir_removed": c.get("merged_dir_removed"),
              "error": (c.get("error") or "")[:300], "traceback_tail": c.get("traceback_tail", [])[-15:],
              "base_texts": c.get("base_texts"), "backend_lines": c.get("backend_lines", [])[:4]}
             for c in vllm],
}, indent=2))